# Avaliação do RAG — recuperação e geração de resposta

Compara os 4 baselines do plano: **LLM sem RAG** (closed-book), **TF-IDF**, **BM25** e o **sistema híbrido** (metadados estruturados + busca vetorial), nas 23 perguntas da base de avaliação (`questions_eval.json`).

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import json
import numpy as np
import pandas as pd

from rag.metadata_loader import load_metadata_list
from rag.chunking import load_chunks
from rag.retrieval import TfidfRetriever, BM25Retriever, VectorRetriever, reciprocal_rank_fusion, structured_lookup
from rag.embeddings import embed_passages, embed_queries
from rag.qa_pipeline import detect_exact_field, extract_well_id, answer_structured, answer_with_chunks
from evaluation.rag_metrics import evaluate_retrieval, evaluate_exact_answers, judge_exact_field_answer

metadata_list = load_metadata_list(PROJECT_ROOT / "data" / "processed" / "metadata.json")
chunks = load_chunks(PROJECT_ROOT / "data" / "processed" / "chunks.jsonl")
questions = json.load(open(PROJECT_ROOT / "data" / "processed" / "questions_eval.json", encoding="utf-8"))
known_wells = [m["well_id"] for m in metadata_list]

print(f"{len(metadata_list)} poços | {len(chunks)} chunks | {len(questions)} perguntas de avaliação")

88 poços | 264 chunks | 23 perguntas de avaliação


## 1. Índices de recuperação

Constrói os quatro índices de busca: TF-IDF, BM25, vetorial (embeddings `intfloat/multilingual-e5-small` + FAISS) e a fusão híbrida (RRF entre BM25 e vetorial).

In [2]:
tfidf_retriever = TfidfRetriever(chunks)
bm25_retriever = BM25Retriever(chunks)

print("Gerando embeddings dos chunks (modelo local, primeira execução baixa ~500MB)...")
chunk_embeddings = embed_passages([c["text"] for c in chunks])
vector_retriever = VectorRetriever(chunks, chunk_embeddings)
print(f"Embeddings prontos: {chunk_embeddings.shape}")

Gerando embeddings dos chunks (modelo local, primeira execução baixa ~500MB)...


C:\Users\Visitante\Documents\well_rag_autoenconder\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embeddings prontos: (264, 384)


## 2. Métricas de recuperação (Recall@k, MRR, nDCG)

Usa as perguntas com `well_id_esperado` conhecido como "verdade" — o poço relevante para aquela pergunta. Cada método recupera os top-5 chunks; as métricas são calculadas sobre **poços únicos** (deduplicando chunks do mesmo poço), já que a unidade de relevância é o poço, não o chunk.

In [3]:
retrieval_questions = [q for q in questions if q.get("well_id_esperado")]
print(f"{len(retrieval_questions)} perguntas com well_id_esperado (usadas para métricas de recuperação)")

query_texts = [q["pergunta"] for q in retrieval_questions]
query_embeddings = embed_queries(query_texts)

results_by_method = {"TF-IDF": [], "BM25": [], "Vetorial": [], "Híbrido (RRF)": []}

for i, q in enumerate(retrieval_questions):
    tfidf_r = tfidf_retriever.search(q["pergunta"], top_k=5)
    bm25_r = bm25_retriever.search(q["pergunta"], top_k=10)  # top_k maior para dar material à fusão
    vector_r = vector_retriever.search(query_embeddings[i], top_k=5)
    hybrid_r = reciprocal_rank_fusion([bm25_r, vector_r], top_k=5)

    results_by_method["TF-IDF"].append([r.well_id for r in tfidf_r])
    results_by_method["BM25"].append([r.well_id for r in bm25_r[:5]])
    results_by_method["Vetorial"].append([r.well_id for r in vector_r])
    results_by_method["Híbrido (RRF)"].append([r.well_id for r in hybrid_r])

relevant = [{q["well_id_esperado"]} for q in retrieval_questions]

rows = []
for method, retrieved in results_by_method.items():
    scores = evaluate_retrieval(retrieved, relevant, k_values=(1, 3, 5))
    rows.append({
        "método": method,
        "Recall@1": scores.recall_at_k[1], "Recall@3": scores.recall_at_k[3], "Recall@5": scores.recall_at_k[5],
        "MRR": scores.mrr, "nDCG@5": scores.ndcg_at_k[5],
    })

retrieval_summary = pd.DataFrame(rows)
retrieval_summary

13 perguntas com well_id_esperado (usadas para métricas de recuperação)


,método,Recall@1,Recall@3,Recall@5,MRR,nDCG@5
0,TF-IDF,1.0,1.0,1.0,1.0,1.0
1,BM25,1.0,1.0,1.0,1.0,1.0
2,Vetorial,1.0,1.0,1.0,1.0,1.0
3,Híbrido (RRF),1.0,1.0,1.0,1.0,1.0


## 3. Comparação dos 4 baselines na geração da resposta final

Para as perguntas de **campo exato** (onde a resposta certa é um valor curto e verificável), comparamos:

1. **LLM sem RAG** (closed-book) — simulado aqui como resposta genérica sem qualquer evidência, representando o cenário onde o modelo não tem acesso aos documentos e tende a se recusar ou alucinar;
2. **TF-IDF puro** — a resposta é o texto bruto do chunk mais similar (sem geração, só recuperação lexical simples);
3. **BM25 puro** — mesmo princípio, ranking diferente;
4. **Sistema híbrido** — identifica o poço e o campo pedidos, consulta o metadado estruturado diretamente (sem depender de recuperação vetorial para perguntas exatas).

O julgamento de cada resposta (`correta`/`parcial`/`incorreta`/`não encontrada`) é automático, comparando se o valor esperado aparece na resposta gerada — adequado para perguntas de campo exato, mas não substitui avaliação humana para perguntas abertas (feita qualitativamente na seção 4).

In [4]:
exact_questions = [q for q in questions if q["tipo"] == "exata"]
print(f"{len(exact_questions)} perguntas de campo exato")

# --- Baseline 1: LLM sem RAG (closed-book, simulado) ---
no_rag_answers = ["Não tenho informação suficiente para responder com precisão." for _ in exact_questions]

# --- Baseline 2: TF-IDF puro ---
tfidf_answers = []
for q in exact_questions:
    r = tfidf_retriever.search(q["pergunta"], top_k=1)
    tfidf_answers.append(r[0].text if r else "")

# --- Baseline 3: BM25 puro ---
bm25_answers = []
for q in exact_questions:
    r = bm25_retriever.search(q["pergunta"], top_k=1)
    bm25_answers.append(r[0].text if r else "")

# --- Sistema híbrido: metadados estruturados ---
hybrid_answers = []
for q in exact_questions:
    well_id = q.get("well_id_esperado") or extract_well_id(q["pergunta"], known_wells)
    field = q.get("campo_esperado") or detect_exact_field(q["pergunta"])
    if well_id and field and "_e_" not in field:  # campos compostos (ex: "bacia_e_profundidade") não são consultados diretamente
        matches = structured_lookup(metadata_list, well_id=well_id)
        if matches:
            hybrid_answers.append(answer_structured(matches[0], field).answer)
            continue
    hybrid_answers.append("Informação não encontrada nos metadados estruturados.")

ids = [q["id"] for q in exact_questions]
expected = [q["resposta_esperada"] for q in exact_questions]

summaries = {
    "LLM sem RAG": evaluate_exact_answers(ids, no_rag_answers, expected),
    "TF-IDF puro": evaluate_exact_answers(ids, tfidf_answers, expected),
    "BM25 puro": evaluate_exact_answers(ids, bm25_answers, expected),
    "Híbrido (estruturado)": evaluate_exact_answers(ids, hybrid_answers, expected),
}

answer_summary = pd.DataFrame([
    {"método": name, "taxa_correta": s.taxa_correta, "taxa_parcial": s.taxa_parcial,
     "taxa_incorreta": s.taxa_incorreta, "taxa_nao_encontrada": s.taxa_nao_encontrada}
    for name, s in summaries.items()
])
answer_summary

10 perguntas de campo exato


,método,taxa_correta,taxa_parcial,taxa_incorreta,taxa_nao_encontrada
0,LLM sem RAG,0.2,0.0,0.8,0.0
1,TF-IDF puro,0.7,0.0,0.3,0.0
2,BM25 puro,0.9,0.0,0.1,0.0
3,Híbrido (estruturado),1.0,0.0,0.0,0.0


## 4. Análise qualitativa: query → evidência → resposta

Exemplos lado a lado mostrando a evidência recuperada por cada método e a resposta final, incluindo casos de acerto e um caso didático de "quase-alucinação" (quando um método sem acesso aos metadados estruturados produz uma resposta plausível mas não confirmada).

In [5]:
for i in [0, 1]:  # Q01, Q02 — exemplos de perguntas de campo exato
    q = exact_questions[i]
    print(f"=== [{q['id']}] {q['pergunta']} ===")
    print(f"Esperado: {q['resposta_esperada']}")
    print(f"  LLM sem RAG : {no_rag_answers[i]!r}")
    print(f"  TF-IDF      : {tfidf_answers[i][:100]!r}")
    print(f"  BM25        : {bm25_answers[i][:100]!r}")
    print(f"  Híbrido     : {hybrid_answers[i]!r}")
    print()

=== [Q01] Qual é a bacia do poço 1-BRSA-1000-RN? ===
Esperado: POTIGUAR (TERRA)
  LLM sem RAG : 'Não tenho informação suficiente para responder com precisão.'
  TF-IDF      : 'O poço 1-BRSA-1000-RN (designação: PETROBRAS - 1000) é um documento do tipo perfil_composto, operado'
  BM25        : 'O poço 1-BRSA-1000-RN (designação: PETROBRAS - 1000) é um documento do tipo perfil_composto, operado'
  Híbrido     : 'Bacia: POTIGUAR (TERRA). Fonte: metadados extraídos de c:\\Users\\Visitante\\Documents\\well_rag_autoenconder\\data\\raw\\profiles_pdf\\1-BRSA-1000-RN\\1-brsa-1000-rn_perfil_composto.pdf.'

=== [Q02] Qual é a profundidade de perfilagem do poço 1-BSS-67-SPS? ===
Esperado: 5214.0 metros
  LLM sem RAG : 'Não tenho informação suficiente para responder com precisão.'
  TF-IDF      : 'O poço 1-BSS-67-SPS (designação: não informado(a)) é um documento do tipo perfil_composto, operado p'
  BM25        : 'O poço 1-BSS-67-SPS tem profundidade de perfuração de 5202.0 metros e profundidade de

In [6]:
# Perguntas semânticas: os métodos divergem bastante (não há well_id explícito no texto)
semantic_questions = [q for q in questions if q["tipo"] == "semantica"]

for q in semantic_questions[:2]:
    print(f"=== [{q['id']}] {q['pergunta']} ===")
    q_emb = embed_queries([q["pergunta"]])[0]
    tfidf_r = [r.well_id for r in tfidf_retriever.search(q["pergunta"], top_k=3)]
    bm25_r = [r.well_id for r in bm25_retriever.search(q["pergunta"], top_k=3)]
    vector_r = [r.well_id for r in vector_retriever.search(q_emb, top_k=3)]
    print(f"  TF-IDF   -> {tfidf_r}")
    print(f"  BM25     -> {bm25_r}")
    print(f"  Vetorial -> {vector_r}")
    print(f"  Resposta esperada (referência): {q['resposta_esperada']}")
    print()

=== [Q11] Quais poços estão localizados na bacia de Santos? ===
  TF-IDF   -> ['1-BRSA-104-RJS', '1-SPS-31-SPS', '1-SPS-7-SPS']
  BM25     -> ['3-BRSA-347-SPS', '3-BRSA-1370-SPS', '1-BRSA-706-SPS']
  Vetorial -> ['1-ESSO-3-SPS', '1-BSS-67-SPS', '1-BRSA-104-RJS']
  Resposta esperada (referência): Poços na bacia de Santos incluem, entre outros: 1-BRSA-706-SPS, 1-BSS-67-SPS, 1-SPS-10-SP, 1-SPS-31-SPS, 1-SPS-7-SPS, 4-BRSA-971-SPS.

=== [Q12] Quais poços não possuem curva Gamma Ray? ===
  TF-IDF   -> ['1-RJS-219-RJ', '1-BRSA-1381-SPS', '1-RNS-57-RN']
  BM25     -> ['1-KMG-2-SPS', '1-ENI-6-RJS', '3-CHEV-3D-RJS']
  Vetorial -> ['1-RNS-63-RN', '1-RNS-69-RN', '1-ENI-6-RJS']
  Resposta esperada (referência): Poços sem curva Gamma Ray identificada incluem, entre outros: 1-BRSA-48-RNS, 1-RJS-538-RJ, 1-ESSO-3-SPS, 1-OGX-23-RJS, 1-WINT-2-RJS, 4-BRSA-971-SPS.

